# Mobiliando a planta: um agente propõe, outro explica, o código decide

No primeiro projeto da oficina, um modelo de linguagem propôs a planta de uma
casa e um verificador escrito em código conferiu a proposta contra regras de
projeto. A última célula daquele notebook listava o que a ferramenta **não**
fazia, e mobiliário estava na lista. Este notebook começa exatamente ali: a
planta já foi aprovada, e agora é preciso pôr os móveis dentro dela.

São dois agentes:

- o **mobiliador** escolhe onde cada móvel fica, a partir de um catálogo com
  dimensões fixas;
- o **fiscal** lê o que o código mediu e redige o parecer em português,
  citando de onde veio cada exigência.

Entre os dois, o código mede: se o móvel cabe no cômodo, se sobra espaço
diante de cada porta, se dá para abrir o guarda-roupa, se uma cadeira de
rodas consegue girar. A tese cabe numa frase: **o modelo escolhe e explica; o
código mede e decide.** O fiscal não consegue aprovar nem reprovar nada, e o
mobiliador não consegue contestar o parecer.

Rode as células de cima para baixo. Nada aqui precisa de terminal.

> Esta é uma ferramenta didática. Ela **não substitui projeto de profissional
> habilitado**, e os parâmetros de circulação embutidos são didáticos e
> provisórios.

## 1. Configuração

As mesmas três informações do primeiro notebook. Elas ficam só nesta sessão:
nada é gravado em arquivo, e some tudo quando você fechar o Codespace.

In [ ]:
import os
from getpass import getpass
from pathlib import Path

import yaml

from floorplan_guardrails.furnished_renderer import draw_furnished, draw_negotiation
from floorplan_guardrails.furnisher import FurnisherError, furnisher_from_env
from floorplan_guardrails.furniture import (
    FurnishedPlan,
    FurnishingProposal,
    FurnishingRequest,
    Profile,
    load_catalog,
)
from floorplan_guardrails.furniture_rules import load_furniture_rules
from floorplan_guardrails.generator import (
    GeneratorError,
    generator_from_env,
    openai_base_url,
)
from floorplan_guardrails.inspection import inspect, precheck
from floorplan_guardrails.inspector import (
    APPROVED_SUMMARY,
    Consultation,
    InspectorError,
    Review,
    inspector_from_env,
    messages_review,
    parameter_of,
)
from floorplan_guardrails.negotiation import negotiate, only_integrity
from floorplan_guardrails.negotiation_log import NegotiationLog, load_negotiation
from floorplan_guardrails.rules import load_rules
from floorplan_guardrails.schema import FloorPlan

# O notebook acha a raiz do repositório subindo as pastas até encontrar o
# arquivo de regras, então funciona independente de onde o kernel foi aberto.
RAIZ = next(
    pasta
    for pasta in (Path.cwd(), *Path.cwd().parents)
    if (pasta / "config" / "rules.yaml").exists()
)
EXEMPLOS = RAIZ / "examples" / "p2"
ARQUIVO_DE_PARAMETROS = RAIZ / "config" / "furniture_rules.yaml"

print("Repositório encontrado em:", RAIZ)

In [ ]:
def pedir(variavel: str, rotulo: str, secreto: bool = False) -> None:
    """Pede um valor e guarda só na sessão, sem gravar em lugar nenhum."""
    if os.environ.get(variavel):
        print(f"{rotulo}: já preenchido nesta sessão.")
        return

    valor = getpass(f"{rotulo}: ") if secreto else input(f"{rotulo}: ")
    os.environ[variavel] = valor.strip()


pedir("AZURE_OPENAI_ENDPOINT", "Endpoint do Azure")

# O portal mostra mais de uma URL para o mesmo recurso, e qualquer uma serve:
# aqui ela é reduzida à forma que o gerador usa. Se não der, o valor é apagado
# para que rodar a célula de novo peça o endpoint outra vez.
try:
    os.environ["AZURE_OPENAI_ENDPOINT"] = openai_base_url(
        os.environ["AZURE_OPENAI_ENDPOINT"]
    )
    print("Endpoint em uso:", os.environ["AZURE_OPENAI_ENDPOINT"])
except GeneratorError as erro:
    del os.environ["AZURE_OPENAI_ENDPOINT"]
    print(erro)
    print("Rode esta célula de novo para colar o endpoint certo.")

pedir("AZURE_OPENAI_DEPLOYMENT", "Nome do deployment")
pedir("AZURE_OPENAI_API_KEY", "Chave de API", secreto=True)

# Esta variável precisa NÃO existir. O caminho /openai/v1 recusa o parâmetro
# api-version, e o cliente a lê do ambiente por conta própria: se ela estiver
# definida, toda chamada falha com erro 400.
os.environ.pop("AZURE_OPENAI_API_VERSION", None)

DEPLOYMENT = os.environ["AZURE_OPENAI_DEPLOYMENT"]

In [ ]:
try:
    gerador = generator_from_env()
    resposta = await gerador.check()
except GeneratorError as erro:
    print("A conexão NÃO funcionou.\n")
    print(erro)
else:
    print("Conexão funcionando. O modelo respondeu:", resposta)

Agora o que o código usa para medir: o **catálogo** de móveis, com as
dimensões de cada um, e os **parâmetros de circulação**, com o quanto de
espaço livre cada verificação exige. Os parâmetros moram em
`config/furniture_rules.yaml`, e nenhum dos dois agentes os vê nas
instruções: o mobiliador só descobre o que é exigido lendo o parecer.

In [ ]:
catalogo = load_catalog(RAIZ / "config" / "furniture_catalog.yaml")
regras_moveis = load_furniture_rules(catalogo, ARQUIVO_DE_PARAMETROS)
regras_planta = load_rules(RAIZ / "config" / "rules.yaml")

try:
    mobiliador = furnisher_from_env()
    fiscal = inspector_from_env(regras_moveis, catalogo)
except (FurnisherError, InspectorError) as erro:
    print(erro)
else:
    print("Mobiliador e fiscal prontos.")

print(f"\nO catálogo tem {len(catalogo)} móveis:")
for item_id, item in catalogo.items():
    print(f"  {item_id}: {item.name}, {item.width:.2f} x {item.depth:.2f} m")

As plantas de exemplo ficam em `examples/p2/plans/`, todas aprovadas pelo
verificador do primeiro projeto. Cada uma tem um pedido de móveis pronto em
`examples/p2/requests/`, com o mesmo nome.

- `casa_4_comodos`: a casa retangular de sala, cozinha, quarto e banheiro.
- `casa_em_l`: sala e cozinha embaixo, quarto e banheiro em cima.
- `quarto_apertado`: o quarto tem 2,40 × 3,40 m, no limite do que o primeiro
  projeto aceita.
- `aprovada_gerada`: uma planta que o modelo desenhou sozinho, a partir de uma
  descrição em português, e que o verificador do primeiro projeto aprovou.

In [ ]:
def carregar_planta(nome: str) -> FloorPlan:
    caminho = EXEMPLOS / "plans" / f"{nome}.json"
    return FloorPlan.model_validate_json(caminho.read_text(encoding="utf-8"))


def carregar_pedido(nome: str) -> FurnishingRequest:
    caminho = EXEMPLOS / "requests" / f"{nome}.yaml"
    return FurnishingRequest.model_validate(
        yaml.safe_load(caminho.read_text(encoding="utf-8"))
    )


def carregar_proposta(nome: str) -> FurnishingProposal:
    caminho = EXEMPLOS / "proposals" / f"{nome}.json"
    return FurnishingProposal.model_validate_json(caminho.read_text(encoding="utf-8"))


def mostrar_violacoes(violacoes) -> None:
    if not violacoes:
        print("Nenhuma violação.")
    for violacao in violacoes:
        print(f"- [{violacao.rule_id}] {violacao.message}")


def mostrar_parecer(parecer: Review) -> None:
    print(parecer.summary)
    for numero, item in enumerate(parecer.feedback, 1):
        print(f"\n{numero}. {item.message}")
        if item.suggestion:
            print("   Sugestão do fiscal, verificada na próxima rodada:")
            print("  ", item.suggestion)


async def parecer_sobre(mobiliada: FurnishedPlan, violacoes, perfil: Profile) -> Review:
    """O mesmo caminho da negociação: o fiscal só é chamado quando precisa."""
    if not violacoes:
        return Review(summary=APPROVED_SUMMARY)
    if only_integrity(violacoes):
        return messages_review(violacoes)
    return await fiscal.review(mobiliada, violacoes, perfil)


for caminho in sorted((EXEMPLOS / "plans").glob("*.json")):
    print("-", caminho.stem)

## 2. Nível 1: mobiliar

Uma chamada ao mobiliador sobre a `casa_4_comodos`, com o pedido pronto. Ele
recebe a planta, a geometria que o código calculou (os limites de cada cômodo
e o vão de cada porta), o catálogo filtrado aos móveis pedidos e o pedido.

Sobre o tempo: cada chamada do mobiliador leva cerca de 15 s. **Não
interrompa a célula.** Enquanto houver um asterisco ao lado dela, o modelo está
trabalhando.

In [ ]:
planta1 = carregar_planta("casa_4_comodos")
pedido_pronto = carregar_pedido("casa_4_comodos")

for comodo, itens in pedido_pronto.items.items():
    print(comodo, itens)

**Exercício.** Mude o pedido na célula abaixo e rode de novo daqui para
baixo. Tire ou acrescente um móvel (os `item_id` estão na lista do catálogo,
lá em cima), ou escreva uma nota em português, como "a cama com a cabeceira
na parede norte".

In [ ]:
pedido1 = FurnishingRequest(
    items=pedido_pronto.items,
    notes="",
)
perfil1 = Profile(accessible=False)

In [ ]:
tentativa = await mobiliador.propose(planta1, catalogo, pedido1, perfil1)
mobiliada1 = FurnishedPlan(plan=planta1, proposal=tentativa.proposal)

print(f"Resposta em {tentativa.latency_ms / 1000:.1f} s.")
print("Notas do mobiliador:", tentativa.proposal.design_notes)

Antes de medir circulação, o código confere se a proposta fecha como desenho:
todo móvel dentro do seu cômodo, nenhum sobre o outro, e o pedido atendido,
sem faltar nem sobrar. Isso é o `precheck`.

In [ ]:
violacoes_desenho = precheck(mobiliada1, catalogo, pedido1)
mostrar_violacoes(violacoes_desenho)

desenho1 = draw_furnished(
    mobiliada1, catalogo, regras_moveis, perfil1, violacoes_desenho
)

## 3. Nível 2: o fiscal

Agora a conferência inteira, com circulação: espaço livre diante de cada
porta, a faixa de uso de cada móvel (o lado da cama por onde se sobe, a frente
do guarda-roupa que se abre) e, quando o morador usa cadeira de rodas, o giro
dentro do cômodo.

Quem mede é o `inspect`. Quando alguma verificação de circulação falha, o
fiscal recebe as violações já medidas e redige o parecer: explica cada uma e
sugere uma correção. Cada chamada do fiscal leva de 20 a 30 s.

Primeiro, sobre a proposta do nível 1.

In [ ]:
violacoes1 = inspect(mobiliada1, catalogo, regras_moveis, perfil1, pedido1)
mostrar_violacoes(violacoes1)

parecer1 = await parecer_sobre(mobiliada1, violacoes1, perfil1)
print()
mostrar_parecer(parecer1)

A proposta do nível 1 pode ter passado. Para garantir uma reprovação, esta é
uma disposição feita à mão com dois erros conhecidos, `layout_ruim.json`: a
cama logo diante da porta do quarto, e o guarda-roupa encostado na parede com
as portas voltadas para ela.

Com `show_zones=True`, o desenho mostra todas as faixas medidas, tracejadas.
As que falharam ficam em vermelho.

In [ ]:
mobiliada_ruim = FurnishedPlan(plan=planta1, proposal=carregar_proposta("layout_ruim"))
pedido_ruim = carregar_pedido("casa_4_comodos")
perfil_padrao = Profile(accessible=False)

violacoes_ruim = inspect(
    mobiliada_ruim, catalogo, regras_moveis, perfil_padrao, pedido_ruim
)
mostrar_violacoes(violacoes_ruim)

desenho_ruim = draw_furnished(
    mobiliada_ruim,
    catalogo,
    regras_moveis,
    perfil_padrao,
    violacoes_ruim,
    show_zones=True,
)

In [ ]:
parecer_ruim = await fiscal.review(mobiliada_ruim, violacoes_ruim, perfil_padrao)
mostrar_parecer(parecer_ruim)

Repare no que veio do código e no que veio do modelo. A mensagem de cada
violação, com o medido e o exigido, é do código. A explicação e a sugestão
são do fiscal. Se ele escrever na explicação um número que o código não
forneceu, o código troca a explicação inteira pela mensagem original: o
modelo não consegue inventar uma medida.

A sugestão pode ter número (`m4 em x = 0.75, y = 4.10, rotação 0`), porque é
uma proposta e não uma medida. Ela só vale depois que o código medir de novo,
na rodada seguinte.

O código também troca o nome interno de um parâmetro, se o fiscal o escrever
(`door_clearance_depth`, por exemplo), pela descrição dele: o nome interno é
para a ferramenta, não para quem lê.

**De onde vem cada exigência.** Toda violação de circulação carrega a fonte
do parâmetro que ela cobrou. As fontes estão no campo `source` de cada
parâmetro, em `config/furniture_rules.yaml`.

O fiscal não sabe os números de cor: para citar a fonte, ele chama uma
ferramenta, `consultar_parametro`, que lê o mesmo arquivo. A célula abaixo
mostra quantas vezes ele a chamou neste parecer e o que a ferramenta devolve
para os parâmetros que estas violações cobram.

In [ ]:
for violacao in violacoes_ruim:
    print(f"[{violacao.rule_id}] fonte: {violacao.source}")

print(f"\nO fiscal chamou consultar_parametro {parecer_ruim.tool_calls} vez(es).")
if parecer_ruim.invalid_params:
    print("Pediu parâmetros que não existem:", parecer_ruim.invalid_params)

# A mesma ferramenta que o fiscal usa, chamada aqui à mão, para os
# parâmetros que estas violações cobram.
ferramenta = Consultation(regras_moveis)
for violacao in violacoes_ruim:
    param_id = parameter_of(violacao, mobiliada_ruim.proposal)
    print(f"\nconsultar_parametro({param_id!r}) devolve:")
    print(" ", ferramenta.consult(param_id))

O mesmo layout, agora para um morador que usa cadeira de rodas. Nada muda na
disposição: muda o perfil, e com ele entra a verificação de giro. O desenho
mostra, em cada cômodo verificado, o maior quadrado livre que o código achou.

In [ ]:
perfil_acessivel = Profile(accessible=True)

violacoes_acessivel = inspect(
    mobiliada_ruim, catalogo, regras_moveis, perfil_acessivel, pedido_ruim
)
mostrar_violacoes(violacoes_acessivel)

desenho_acessivel = draw_furnished(
    mobiliada_ruim,
    catalogo,
    regras_moveis,
    perfil_acessivel,
    violacoes_acessivel,
    show_zones=True,
)

In [ ]:
parecer_acessivel = await fiscal.review(
    mobiliada_ruim, violacoes_acessivel, perfil_acessivel
)
mostrar_parecer(parecer_acessivel)

## 4. Nível 3: a negociação

Agora o ciclo inteiro, sem parar entre um passo e outro: o mobiliador propõe,
o código mede, o fiscal redige o parecer, e o parecer volta ao mobiliador,
que devolve a disposição inteira corrigida. A negociação termina de um de
quatro jeitos:

- **aprovada**: nenhuma verificação falhou;
- **desistência**: o mobiliador declarou que algum móvel não cabe;
- **não convergiu**: as rodadas acabaram com violações;
- **inviável**: antes da primeira rodada, o código viu que algum cômodo não
  comporta o giro de cadeira de rodas nem vazio. Nenhum modelo é chamado, e a
  planta precisa voltar à etapa de geração.

Uma boa negociação não é a que sempre aprova. É a que termina num destes
três lugares: **aprovar, desistir ou reconhecer o limite**. Um móvel que não
cabe, dito com todas as letras, vale mais do que uma planta aprovada à força.

Uma negociação inteira leva normalmente de 1 a 2 minutos; em cerca de 1 a
cada 10 execuções, passa de 2 minutos. **Não interrompa a célula.**

In [ ]:
async def negociar(nome: str, perfil: Profile):
    """Negocia a planta de exemplo com o seu pedido e conta como foi."""
    negociacao = await negotiate(
        carregar_planta(nome),
        carregar_pedido(nome),
        perfil,
        mobiliador,
        fiscal,
        catalogo,
        regras_moveis,
        log=NegotiationLog.create(RAIZ / "runs" / "p2"),
        deployment=DEPLOYMENT,
        plan_rules=regras_planta,
    )

    if negociacao.status == "infeasible":
        print("Estado final: infeasible. Nenhum modelo foi chamado.\n")
        mostrar_violacoes(negociacao.violations)
        return negociacao

    print(f"Estado final: {negociacao.status}, em {len(negociacao.rounds)} rodadas.")

    for numero, rodada in enumerate(negociacao.rounds, 1):
        print(f"\n--- Rodada {numero}: {rodada.status} ---")
        if rodada.status == "declined":
            for omissao in rodada.proposal.omissions:
                print(f"- {omissao.item_id} em {omissao.room_id}: {omissao.reason}")
            continue
        mostrar_violacoes(rodada.violations)
        if rodada.violations and rodada.review is not None:
            print()
            mostrar_parecer(rodada.review)

    return negociacao

Primeiro, a casa de 4 cômodos, com o perfil padrão.

In [ ]:
nivel3 = await negociar("casa_4_comodos", Profile(accessible=False))

In [ ]:
desenho3 = draw_negotiation(nivel3.history, catalogo, regras_moveis, nivel3.profile)

### Exercícios

Cada exercício tem a sua célula já preenchida. Rode na ordem, e depois troque
a planta ou o perfil para ver o que muda.

**1. A casa em L, com cadeira de rodas.** O banheiro da `casa_em_l` tem
1,20 m de largura. Ela foi aprovada no primeiro projeto, mas não serve para
quem usa cadeira de rodas: a resposta vem na hora, sem chamar modelo.

In [ ]:
exercicio1 = await negociar("casa_em_l", Profile(accessible=True))

desenho_ex1 = draw_negotiation(
    exercicio1.history, catalogo, regras_moveis, exercicio1.profile
)

**2. O quarto apertado.** O quarto da `quarto_apertado` tem 2,40 × 3,40 m, e o
pedido é cama de casal e guarda-roupa. O mobiliador consegue, ou desiste?

Na medição da Fase 5 (`docs/avaliacao-p2.md`), 1 de 2 negociações desta
planta com o perfil padrão terminou em desistência, e a outra sem acordo.
O seu resultado pode ser outro: o modelo varia de uma execução para
outra, como a turma viu na oficina 1.

In [ ]:
exercicio2 = await negociar("quarto_apertado", Profile(accessible=False))

desenho_ex2 = draw_negotiation(
    exercicio2.history, catalogo, regras_moveis, exercicio2.profile
)

**3. Uma planta com cadeira de rodas.** Agora o giro de cadeira de rodas
entra em cada quarto, banheiro e sala. Troque a planta e compare.

Na medição da Fase 5 (`docs/avaliacao-p2.md`), 2 de 2 negociações da
`casa_4_comodos` com cadeira de rodas terminaram sem acordo; somando as
quatro plantas, 7 de 8. O seu resultado pode ser outro: o modelo varia de uma
execução para outra, como a turma viu na oficina 1.

In [ ]:
exercicio3 = await negociar("casa_4_comodos", Profile(accessible=True))

desenho_ex3 = draw_negotiation(
    exercicio3.history, catalogo, regras_moveis, exercicio3.profile
)

**4. Uma sugestão errada do fiscal.** Esta negociação foi gravada durante a
medição da Fase 5, e é desenhada aqui sem chamar o modelo. Na rodada 1, o
fiscal sugeriu uma posição para o guarda-roupa. O mobiliador seguiu a
sugestão ao pé da letra, e na rodada 2 o código mediu o resultado.

In [ ]:
sugestao_errada = load_negotiation(EXEMPLOS / "replays" / "sugestao-errada.jsonl")
primeira, segunda = sugestao_errada.rounds[0], sugestao_errada.rounds[1]

print("Sugestões do fiscal na rodada 1:")
for item in primeira.review.feedback:
    print("-", item.suggestion)

print("\nO que o código mediu na rodada 2:")
mostrar_violacoes(segunda.violations)

desenho_ex4 = draw_negotiation(
    sugestao_errada.history, catalogo, regras_moveis, sugestao_errada.profile
)

**Para debater.** O fiscal recebeu a geometria de todos os cômodos, a
convenção de rotação e os números medidos, e mesmo assim sugeriu pôr o
guarda-roupa diante da porta. Por que o código confere de novo cada sugestão
do fiscal, em vez de aceitá-la? Na medição da Fase 5
(`docs/avaliacao-p2.md`), o mobiliador seguiu 57 de 66 sugestões ao pé da
letra, e 30 dessas 57 ainda falharam na rodada seguinte.

E a pergunta do outro lado: o mobiliador também não pode contestar o
parecer, só reposicionar os móveis ou declarar que algum não cabe. Por que
não deixar um modelo convencer o outro?

### Para ir além: mudando a exigência

Abra `config/furniture_rules.yaml` no editor, ao lado deste notebook, e mude
um número: o `value` de `turning_diameter`, ou uma das `values` de
`use_zone_depth` (a do guarda-roupa, `wardrobe`, por exemplo). Salve o
arquivo, rode a célula abaixo para recarregar os parâmetros, e depois rode de
novo qualquer uma das negociações acima.

O mobiliador continua sem ver número nenhum. Ele só vai perceber a mudança
pelo parecer.

In [ ]:
regras_moveis = load_furniture_rules(catalogo, ARQUIVO_DE_PARAMETROS)

# O fiscal guarda os parâmetros que recebeu, então é montado de novo.
try:
    fiscal = inspector_from_env(regras_moveis, catalogo)
except InspectorError as erro:
    print(erro)

print("Diante de porta:", regras_moveis.door_clearance_depth.value, "m")
print("Giro de cadeira de rodas:", regras_moveis.turning_diameter.value, "m")
print("Faixas de uso:")
for item_id, valor in regras_moveis.use_zone_depth.values.items():
    print(f"  {item_id}: {valor} m")

## 5. Replay: negociações gravadas

Toda negociação fica gravada, uma linha por rodada. Estas três foram gravadas
na medição da Fase 5 e ficam em `examples/p2/replays/`. A célula abaixo
desenha uma delas **sem chamar o modelo**. Se a conexão cair ou ficar lenta
durante a oficina, é por aqui que a gente continua.

- `aprovada`: o box bloqueia a porta do banheiro, o fiscal sugere onde
  pô-lo, e a rodada 2 aprova;
- `desistencia`: o quarto apertado com cadeira de rodas, em que o mobiliador
  desiste do guarda-roupa na rodada 2;
- `sugestao-errada`: a negociação do exercício 4.

In [ ]:
# Troque o nome para desenhar outra gravação.
GRAVACAO = "aprovada"

gravada = load_negotiation(EXEMPLOS / "replays" / f"{GRAVACAO}.jsonl")
print(f"{GRAVACAO}: {gravada.status}, {len(gravada.rounds)} rodadas.")

desenho_replay = draw_negotiation(
    gravada.history, catalogo, regras_moveis, gravada.profile
)

## 6. Fechamento

O que você viu:

- um agente escolhendo onde cada móvel fica, a partir de um catálogo com
  medidas fixas;
- o código medindo cada disposição e escrevendo, em português, o que falhou e
  quanto faltou;
- outro agente explicando o que o código mediu, sem poder mudar o veredito;
- e a exigência mudando de lugar quando o arquivo de parâmetros muda.

O que a ferramenta **não** faz:

- folha de porta, sentido de abertura e arco de varredura: a porta aqui é só
  um vão na parede;
- conferir se um móvel bloqueia uma janela;
- móveis que não sejam retangulares, ou girados fora de 0, 90, 180 e 270
  graus;
- rota acessível entre cômodos: o giro de cadeira de rodas é verificado só
  dentro de cada cômodo;
- mobiliário mínimo obrigatório por norma: quem diz que móveis entram é o
  pedido;
- altura, pé-direito e bancadas em L.

**Uma aproximação que vale conhecer.** A norma fala em um círculo de giro. O
código procura um quadrado livre com lado igual ao diâmetro do círculo: se o
quadrado cabe, o círculo cabe dentro dele. É uma aproximação conservadora,
porque o quadrado pede mais espaço do que o círculo. Pode acontecer de o
código reprovar um cômodo em que o círculo caberia, mas nunca de aprovar um em
que ele não cabe. A busca também anda numa grade de 5 cm, e um quadrado que só
caberia fora da grade não é encontrado.

Os parâmetros de circulação são didáticos e provisórios; o campo `source` de
cada um diz de onde veio o número e até onde ele foi conferido. Se você é de
engenharia civil ou arquitetura: estes números servem para mostrar o método,
não para conferir projeto. Acessibilidade segue a ABNT NBR 9050 e a legislação
do lugar da obra, e responde a um profissional habilitado.

Cada negociação ficou gravada em `runs/p2/`, uma linha por rodada, caso você
queira olhar depois o que foi proposto e o que foi cobrado.